# D3-03 Prospective GWP100 across years and scenarios

This notebook keeps one tomato product system fixed and reevaluates it with year- and scenario-specific climate characterization factors.

We will:

1. inspect the packaged prospective method;
2. map the inventory once;
3. reevaluate four representative MESSAGE scenarios;
4. interpret what changes—and what does not.

## Learning goals

After this notebook, you should be able to:

- find scenario and year parameters in an `edges` method;
- distinguish direct emissions from upstream supply-chain emissions;
- reuse one mapped inventory with repeated `evaluate_cfs()` calls;
- compare prospective scores without implying that the inventory itself changed.

## Background

The packaged `Prospective_GWP100.json` method cites Watanabe and Cherubini (2026) and IPCC (2021). It keeps the CO2 CF fixed at 1 and supplies prospective CH4 and N2O CFs by IAM scenario and year.

This notebook uses four MESSAGE pathways spanning very low to high forcing. It does **not** calculate every scenario in the method.

## 1) Select and inspect the tomato activity

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import bw2data as bd
import edges
from edges import EdgeLCIA

### 1.1 Find exactly one activity

The functional unit is **1 kg of Swiss early-harvest greenhouse tomatoes**.

In [ ]:
bd.projects.set_current("aalborg-rlcia-2026")
bafu = bd.Database("bafu")

TOMATO_NAME = (
    "Tomatoes conventional, hors-sol heated, "
    "at greenhouse, early harvest"
)
TOMATO_PRODUCT = TOMATO_NAME

matches = [
    activity
    for activity in bafu
    if activity.get("name") == TOMATO_NAME
    and activity.get("reference product") == TOMATO_PRODUCT
    and activity.get("location") == "CH"
]
if len(matches) != 1:
    raise ValueError(f"Expected one tomato activity; found {len(matches)}.")

tomato_activity = matches[0]
print(tomato_activity)
print("Functional unit: 1", tomato_activity.get("unit"))

### 1.2 Inspect direct climate-relevant emissions

The foreground activity emits fossil CO2 and N2O directly. It has no direct methane exchange; methane enters later through upstream suppliers.

In [ ]:
FOCUS_FLOWS = {
    "Carbon dioxide, fossil",
    "Methane, fossil",
    "Methane, non-fossil",
    "Dinitrogen monoxide",
}

direct_rows = [
    {
        "flow": exchange.input["name"],
        "amount [kg]": float(exchange["amount"]),
    }
    for exchange in tomato_activity.biosphere()
    if exchange.input["name"] in FOCUS_FLOWS
]
direct_emissions = (
    pd.DataFrame(direct_rows)
    .groupby("flow", as_index=False)["amount [kg]"]
    .sum()
)
direct_emissions.style.format({"amount [kg]": "{:.3e}"})

## 2) Inspect the prospective method

### 2.1 Load the JSON file installed with `edges`

In [ ]:
method_path = (
    Path(edges.__file__).resolve().parent
    / "data"
    / "Prospective_GWP100.json"
)
method = json.loads(method_path.read_text())

first_scenario = next(iter(method["scenarios"]))
available_years = sorted(
    int(year)
    for year in method["scenarios"][first_scenario]["CF_CH4"]
)

print("Method:", method["name"])
print("Unit:", method["unit"])
print("Scenarios:", len(method["scenarios"]))
print("Years:", min(available_years), "to", max(available_years))

### 2.2 See which values are fixed or scenario-dependent

The JSON uses parameter names such as `CF_CH4` and `CF_N2O`. `evaluate_cfs()` replaces them with the values for the selected scenario and year.

In [ ]:
rule_summary = pd.DataFrame(
    [
        {
            "flow": exchange["supplier"]["name"],
            "value in method": exchange["value"],
        }
        for exchange in method["exchanges"]
        if exchange["supplier"]["name"] in FOCUS_FLOWS
        and exchange["supplier"].get("categories") == ["air"]
    ]
)
rule_summary.drop_duplicates().reset_index(drop=True)

### 2.3 Choose four pathways and five years

The subset spans very low, low, medium, and high forcing. Five 20-year steps keep the calculation fast and the plot readable.

In [ ]:
scenario_info = pd.DataFrame(
    [
        ("MESSAGE_-_SSP1-VL", "SSP1-VL", "very low", "#2A9D8F"),
        ("MESSAGE_-_SSP1-L", "SSP1-L", "low", "#457B9D"),
        ("MESSAGE_-_SSP2-M", "SSP2-M", "medium", "#F4A261"),
        ("MESSAGE_-_SSP5-H", "SSP5-H", "high", "#E76F51"),
    ],
    columns=["scenario", "pathway", "forcing", "color"],
).set_index("scenario")
selected_scenarios = scenario_info.index.tolist()

missing = [
    scenario
    for scenario in selected_scenarios
    if scenario not in method["scenarios"]
]
if missing:
    raise ValueError(f"Missing scenarios: {missing}")

scenario_info.reset_index()[["scenario", "pathway", "forcing"]]

The selected years must be available for every pathway.

In [ ]:
years = [2020, 2040, 2060, 2080, 2100]

missing_years = {
    scenario: [
        year
        for year in years
        if str(year) not in method["scenarios"][scenario]["CF_CH4"]
    ]
    for scenario in selected_scenarios
}
missing_years = {
    scenario: values
    for scenario, values in missing_years.items()
    if values
}
if missing_years:
    raise ValueError(f"Missing years: {missing_years}")

print(
    f"{len(selected_scenarios)} scenarios × {len(years)} years "
    f"= {len(selected_scenarios) * len(years)} calculations"
)

## 3) Map the inventory once

Only characterization factors change between cases. The technosphere and biosphere inventories therefore need to be calculated and mapped only once.

In [ ]:
prospective_lca = EdgeLCIA(
    demand={tomato_activity: 1},
    filepath=str(method_path),
)
prospective_lca.lci()
prospective_lca.map_exchanges()

print("Inventory calculated and exchanges mapped.")

### Inspect one baseline result

The chart below uses the complete supply chain. It shows why prospective methane CFs matter even though methane is absent from the direct foreground emissions.

In [ ]:
baseline_scenario = "MESSAGE_-_SSP1-VL"
baseline_year = "2020"

prospective_lca.evaluate_cfs(
    scenario=baseline_scenario,
    scenario_idx=baseline_year,
)
prospective_lca.lcia()

cf_table = prospective_lca.generate_cf_table(
    include_unmatched=False
).copy()
flow_contributions = (
    cf_table.groupby("supplier name", as_index=False)["impact"]
    .sum()
)
assert np.isclose(
    flow_contributions["impact"].sum(),
    prospective_lca.score,
)

print(f"Baseline score: {prospective_lca.score:.4f} kg CO2-eq/kg")

In [ ]:
top_flows = (
    flow_contributions.assign(
        magnitude=lambda frame: frame["impact"].abs()
    )
    .nlargest(6, "magnitude")
    .sort_values("impact")
)
top_flows["share [%]"] = (
    top_flows["impact"] / prospective_lca.score * 100
)
axis = top_flows.plot.barh(
    x="supplier name",
    y="impact",
    legend=False,
    figsize=(8, 4.2),
    color="#457B9D",
)
axis.set_title("Largest baseline GWP100 contributions")
axis.set_xlabel("kg CO2-eq / kg tomatoes")
axis.set_ylabel("")
for bar, share in zip(axis.patches, top_flows["share [%]"]):
    axis.text(
        bar.get_width() + prospective_lca.score * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{share:.1f}%",
        va="center",
    )
axis.set_xlim(0, top_flows["impact"].max() * 1.14)
plt.tight_layout()
plt.show()

## 4) Reevaluate the scenario-year combinations

The loop changes only the scenario and year passed to `evaluate_cfs()`. The same mapped inventory is reused for all 20 calculations.

In [ ]:
rows = []

for scenario, info in scenario_info.iterrows():
    parameters = method["scenarios"][scenario]
    for year in years:
        prospective_lca.evaluate_cfs(
            scenario=scenario,
            scenario_idx=str(year),
        )
        prospective_lca.lcia()
        rows.append(
            {
                "scenario": scenario,
                "pathway": info["pathway"],
                "forcing": info["forcing"],
                "year": year,
                "CH4 CF": float(parameters["CF_CH4"][str(year)]),
                "N2O CF": float(parameters["CF_N2O"][str(year)]),
                "score": float(prospective_lca.score),
            }
        )

results = pd.DataFrame(rows)
print(f"Calculated {len(results)} prospective scores.")

### Compare CF trajectories with the resulting score

The first two panels show the changing inputs. The third shows their combined effect on the fixed tomato product system.

In [ ]:
plot_specs = [
    ("CH4 CF", "Prospective CH4 CF", "kg CO2-eq / kg CH4"),
    ("N2O CF", "Prospective N2O CF", "kg CO2-eq / kg N2O"),
    ("score", "Tomato GWP100 score", "kg CO2-eq / kg tomatoes"),
]
figure, axes = plt.subplots(1, 3, figsize=(15, 4.4), sharex=True)

for scenario, info in scenario_info.iterrows():
    data = results.loc[results["scenario"] == scenario]
    for axis, (column, title, ylabel) in zip(axes, plot_specs):
        axis.plot(
            data["year"],
            data[column],
            marker="o",
            linewidth=2,
            color=info["color"],
            label=info["pathway"],
        )
        axis.set_title(title)
        axis.set_xlabel("Year")
        axis.set_ylabel(ylabel)
        axis.grid(alpha=0.25)

handles, labels = axes[0].get_legend_handles_labels()
figure.legend(
    handles,
    labels,
    loc="lower center",
    ncol=4,
    frameon=False,
)
plt.tight_layout(rect=(0, 0.1, 1, 1))
plt.show()

## Quick checkpoint

Before running the next cell, predict:

- which pathway has the highest score in 2100;
- whether the ordering follows CH4, N2O, or both;
- which model component stayed fixed throughout the sweep.

Then compare your prediction with the concise endpoint table.

In [ ]:
comparison = (
    results.loc[results["year"].isin([2020, 2100])]
    .pivot(index="scenario", columns="year", values="score")
    .loc[selected_scenarios]
)
comparison["change"] = comparison[2100] - comparison[2020]
comparison["change [%]"] = (
    comparison["change"] / comparison[2020] * 100
)
comparison.insert(
    0,
    "pathway",
    [
        scenario_info.loc[scenario, "pathway"]
        for scenario in comparison.index
    ],
)
comparison = comparison.reset_index(drop=True).sort_values(2100)
comparison.columns.name = None

In [ ]:
display(
    comparison.style.hide(axis="index").format(
        {
            2020: "{:.4f}",
            2100: "{:.4f}",
            "change": "{:+.4f}",
            "change [%]": "{:+.1f}",
        }
    )
)
print(
    "Spread between highest and lowest 2100 scores:",
    f"{comparison[2100].max() - comparison[2100].min():.4f}",
)

## Recap

You have now:

- loaded and inspected a prospective `edges` method;
- selected four representative MESSAGE pathways and five years;
- calculated and mapped one tomato inventory once;
- shown that direct emissions and full supply-chain emissions are different;
- reevaluated 20 CF cases without rebuilding the inventory;
- compared scenario effects while keeping the activity and functional unit fixed.

In D3-04, the focus shifts from changing CFs to tracing intermediate product flows.